# 1. Dados: seleção, coleta e limpeza

## O fio da história (os 5 notebooks)
Agent skills são arquivos `SKILL.md` com instruções em linguagem natural que um agente de IA de programação segue com as permissões do usuário. Não existe um registro central (como npm ou PyPI): as skills se espalham **copiando arquivos** entre repositórios. Isso levanta três perguntas, uma por notebook:

1. **Notebook 02 (RQ1):** o ecossistema está crescendo, e quem escreve as skills, pessoas ou os próprios agentes?
2. **Notebook 03 (RQ3):** quanto as skills são copiadas, e as cópias atravessam donos diferentes?
3. **Notebook 04 (RQ5):** as skills dão poderes perigosos ao agente, e as cópias editadas aumentam esses poderes?
4. **Notebook 05:** as regras que detectam esses poderes são confiáveis?

**Este notebook** prepara o terreno: de onde vêm os dados, o que conta como "skill" e onde os dados são confiáveis.

---

### English summary

**Methodology step 2 (MSR): selection and collection of data.**

* **Source:** GitSkills, July 2026 snapshot (Zenodo 10.5281/zenodo.21875637): every `SKILL.md` file found by GitHub code search, with repository metadata, bundled files and sampled commit history.
* **Units of analysis:** an *occurrence* is one file at one repository path (used for adoption and reuse); a *skill* is one distinct content, identified by its git blob hash (used for text and security analyses).
* **Selection criteria:** the *strict* population keeps skills with the exact `SKILL.md` basename, valid front matter, a verified content hash, no symlink, and a first commit on or after October 2025, when the format was released.
* **Quality validation:** we report how many skills each filter removes and how complete the commit history is per location class.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
%matplotlib inline
print("database:", os.environ["GITSKILLS_DB"])

## 1.1 Tamanho da base
**Objetivo:** confirmar que estamos usando a base completa, não a amostra.

In [ ]:
counts = db.table_counts()
counts

**O que os dados mostram:** 3.797.117 arquivos `SKILL.md` (ocorrências) em 282.200 repositórios, mais 7,3 milhões de arquivos que vêm junto com as skills (scripts, referências). É a base completa do snapshot de julho/2026.

## 1.2 Carregar as tabelas processadas
**Objetivo:** separar as duas unidades de análise. Uma **ocorrência** é um arquivo em um repositório (mede adoção e reuso); uma **skill distinta** é um texto único, identificado pelo hash (mede conteúdo e segurança). Se o mesmo texto está em 500 repositórios, é 1 skill com 500 cópias.

`scripts/build_features.py` streams the database once and writes one row per skill (with features) and one row per occurrence.

In [ ]:
skills = clean.load_skills()
occ = clean.load_occurrences()
print(f"{len(skills):,} distinct skills, {len(occ):,} occurrences, {occ.repo_full_name.nunique():,} repositories")

**O que os dados mostram:** 3,8 milhões de arquivos correspondem a só 1,88 milhão de textos diferentes: **metade dos arquivos repete um texto que já existe em outro lugar**. Isso **não** quer dizer que a maioria das skills é copiada. Na seção 1.6 veremos que 79% dos **textos** aparecem uma vez só, e que as cópias se concentram em poucos textos muito populares. Contar arquivos ou contar textos dá respostas diferentes, e as duas são verdadeiras.

## 1.3 Critérios de seleção: a população estrita
**Objetivo:** a base é propositalmente inclusiva e guarda arquivos que não são skills válidas. Definimos uma população conservadora que segue a especificação oficial e mostramos quanto cada filtro remove (cada linha aplica mais um filtro sobre a anterior).

Filtros: nome exato `SKILL.md`; frontmatter YAML válido (de onde vêm `name` e `description`); hash do conteúdo verificado; primeiro commit a partir de out/2025 (lançamento do formato); não ser symlink.

In [ ]:
pop = clean.population_report(skills)
save_table(pop, "eda_population_filters")
pop

**O que os dados mostram:** ficam **1.610.839 skills (85,8%)**. Quase todo o corte vem de dois filtros: frontmatter inválido (−12,3%) e nome diferente de `SKILL.md` (−1,9%). Os outros três removem só 118 skills: os dados são consistentes. As 244.470 skills sem `name` caem todas no filtro de frontmatter.

**Por que importa:** todas as análises de conteúdo (notebooks 02–05) usam esta população, então os resultados valem para skills que seguem o padrão.

## 1.4 Onde os arquivos ficam
**Objetivo:** saber em que pasta as skills ficam, porque isso define a qualidade dos dados de histórico (próxima seção). `canonical` = `.claude/skills/<nome>/SKILL.md`; `skills-dir` = outra pasta `skills/`; `other` = qualquer outro lugar.

In [ ]:
occ["location_class"].value_counts(normalize=True).rename("share").to_frame()

In [ ]:
occ["filename"].value_counts().head(10).rename("files").to_frame()

**O que os dados mostram:** só **9,8%** dos arquivos estão no caminho do Claude Code (`canonical`); 55,4% estão em outras pastas `skills/` e 34,8% em outros lugares. O formato já não é exclusivo de uma ferramenta. Variações do nome (`skill.md`, `Skill.md`) somam ~54 mil arquivos e são excluídas da população estrita.

## 1.5 Qualidade: cobertura do histórico de commits
**Objetivo:** os autores do dataset não baixaram o histórico de commits de todos os arquivos, só de uma amostra. Precisamos saber **onde as datas e os autores são confiáveis** antes de analisar tempo e autoria.

In [ ]:
cov = adoption.history_coverage(clean.strict(skills))
save_table(cov, "rq1_history_coverage")
cov

**O que os dados mostram:** o histórico cobre **99,6%** das skills em `canonical`, mas só **~11%** nas outras pastas. **Consequência para o notebook 02:** a curva de crescimento usa só `canonical`, e a autoria por IA vale para as 396.247 skills com histórico, não para todas.

## 1.6 Visão exploratória: cópias por skill e tamanho do texto
**Objetivo:** ver a forma das distribuições antes de medir. Esquerda: quantas skills têm 1, 2, 10, 100… cópias (eixos em log). Direita: tamanho do texto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 2.6))
vc = skills["copies"].value_counts().sort_index()
axes[0].loglog(vc.index, vc.values, ".", ms=3)
axes[0].set(xlabel="copies of a distinct skill", ylabel="# skills")
b = skills["body_chars"].dropna(); b = b[b > 0]
axes[1].hist(np.log10(b), bins=50)
axes[1].set(xlabel="log10(body characters)", ylabel="# skills")
plt.tight_layout(); plt.show()

**Textos × arquivos.** A tabela abaixo liga os dois jeitos de contar: quantos **textos** são únicos ou copiados, e quantos **arquivos** cada grupo ocupa.

In [ ]:
single = skills["copies"] == 1
view = pd.DataFrame({
    "distinct texts": [int(single.sum()), int((~single).sum())],
    "files (occurrences)": [int(skills.loc[single, "copies"].sum()), int(skills.loc[~single, "copies"].sum())],
}, index=["text appears once", "text has copies"])
view.loc["total"] = view.sum()
view["% of texts"] = (100 * view["distinct texts"] / view.loc["total", "distinct texts"]).round(1)
view["% of files"] = (100 * view["files (occurrences)"] / view.loc["total", "files (occurrences)"]).round(1)
print(f"files that repeat an existing text: {len(occ) - len(skills):,} ({1 - len(skills) / len(occ):.1%})")
view

**O que os dados mostram:**

*Gráfico da esquerda (cópias por skill, eixos em log).* Cada ponto diz quantos textos têm exatamente *x* cópias. Os pontos formam quase uma reta descendente: **cauda longa**. **79%** dos textos aparecem uma vez só, e poucos são copiados centenas de vezes (o máximo é 1.409).

*Por que isso não contradiz a seção 1.2:* os 21% de textos copiados ocupam **61%** dos arquivos, porque cada um aparece muitas vezes (em média ~6 arquivos, alguns centenas). A maioria dos **textos** é única, mas a maioria dos **arquivos** é cópia. É como uma biblioteca em que quase todos os títulos têm um exemplar, mas alguns best-sellers com centenas de exemplares ocupam metade das prateleiras.

*Gráfico da direita (tamanho do texto).* Histograma do tamanho do corpo do `SKILL.md` (sem o cabeçalho), em log10 de caracteres: 2 = 100, 3 = 1.000, 4 = 10.000 caracteres. O log é necessário porque os tamanhos vão de dezenas a centenas de milhares de caracteres. A forma é de sino:
* o pico fica entre ~6.000 e ~7.700 caracteres, e a **mediana é 4.601 caracteres** (umas 2 páginas);
* 80% das skills têm menos de 10.000 caracteres; só 11% têm menos de 1.000;
* nas pontas, ~1% tem menos de 100 caracteres (stubs, testes) e ~1% passa de 41.000 (manuais inteiros).

Leitura: a skill típica não é um atalho de uma linha, e sim um documento de algumas páginas que o agente lê e segue. Em um texto desse tamanho, uma instrução sensível editada no meio passa despercebida por quem copia, o que motiva a RQ5. O gráfico usa as 1,88 M skills distintas; as 92 com corpo vazio ficam de fora (não há log de zero).

## Conclusão do notebook 01
* Trabalhamos com **1.610.839 skills** que seguem a especificação (85,8% da base).
* O reuso por cópia é massivo: metade dos arquivos é cópia de outro.
* Datas e autores só são confiáveis em `.claude/skills/` (99,6%); no resto, ~11%. Toda análise temporal e de autoria herda essa limitação.

**Próximo passo:** com a população definida, o notebook 02 mede o crescimento e a autoria.